# MPB LaTeX OCR - Local Im2LaTeX Sample Check

Use this after exporting `im2latex_sample_bundle.zip` from `notebooks/kaggle_train.ipynb`. Extract the zip locally to `data/im2latex_sample_bundle`, then run this notebook to inspect the exact Kaggle images and optionally rerun the local checkpoint on those same crops.

In [ ]:
from __future__ import annotations

import json
import os
import re
import subprocess
import sys
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image, ImageDraw


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "mpb_latex_ocr").exists():
            return candidate
    raise FileNotFoundError("Could not find mpb-latex-ocr project root.")


def select_subprocess_python(project_dir: Path) -> Path:
    venv_python = project_dir / ".venv" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    return venv_python if venv_python.exists() else Path(sys.executable)


PROJECT_DIR = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_DIR / "src"
sys.path.insert(0, str(SRC_DIR))
os.environ["PYTHONPATH"] = f"{SRC_DIR}{os.pathsep}{os.environ.get('PYTHONPATH', '')}"
PYTHON_EXE = select_subprocess_python(PROJECT_DIR)

RUN_DIR = PROJECT_DIR / "outputs"
BUNDLE_DIR = PROJECT_DIR / "data" / "im2latex_sample_bundle"
BUNDLE_MANIFEST = BUNDLE_DIR / "manifest.csv"
EXPORTED_PREDICTIONS_PATH = BUNDLE_DIR / "predictions.jsonl"
BUNDLE_METADATA_PATH = BUNDLE_DIR / "metadata.json"

CHECKPOINT_DIR = RUN_DIR / "checkpoints"
TOKENIZER_PATH = RUN_DIR / "tokenizer.json"
RESOLVED_CONFIG_PATH = RUN_DIR / "resolved_config.json"
LOCAL_OUTPUT_DIR = RUN_DIR / "im2latex_bundle_check"
LOCAL_PREDICTIONS_PATH = LOCAL_OUTPUT_DIR / "test_predictions.jsonl"
LOCAL_CDM_JSON_PATH = LOCAL_OUTPUT_DIR / "cdm_predictions.json"

RUN_LOCAL_EVAL = True

# Optional direct local dataset download/build path. This avoids rerunning the Kaggle notebook.
KAGGLE_DATASET_SLUG = "gregoryeritsyan/im2latex-230k"
KAGGLE_IMAGE_PATH_PREFIX = "/kaggle/input/datasets/gregoryeritsyan/im2latex-230k"
LOCAL_KAGGLE_DATASET_DIR = PROJECT_DIR / "data" / "kaggle" / "im2latex-230k"
DOWNLOAD_KAGGLE_DATASET = False
BUILD_BUNDLE_FROM_LOCAL_DATASET = False
DIRECT_BUNDLE_SAMPLE_COUNT = 12000
DIRECT_BUNDLE_SAMPLE_MODE = "random"  # first, random, best-render, worst-render, prediction-render-fail, render-fail, exact-mismatch
DIRECT_BUNDLE_SAMPLE_SEED = 42
VISUAL_SAMPLE_COUNT = 8
VISUAL_SAMPLE_MODE = "random"  # random, worst_render, best_render, first
VISUAL_SEED = 42

RUN_OFFICIAL_CDM = True  # computes official CDM if metrics are missing
FORCE_RECOMPUTE_OFFICIAL_CDM = False
OFFICIAL_CDM_POOLS = 4
OFFICIAL_CDM_OUTPUT_DIR = LOCAL_OUTPUT_DIR / "official_cdm_windows"


def run_module(module: str, args: list[object], cwd: Path = PROJECT_DIR) -> None:
    command = [str(PYTHON_EXE), "-m", module, *[str(arg) for arg in args]]
    printable = " ".join(f'"{part}"' if " " in part else part for part in command)
    print(f"$ {printable}")
    completed = subprocess.run(
        command,
        cwd=cwd,
        check=False,
        env=os.environ.copy(),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()




def run_command(command: list[object], cwd: Path = PROJECT_DIR) -> None:
    command = [str(part) for part in command]
    printable = " ".join(f'"{part}"' if " " in part else part for part in command)
    print(f"$ {printable}")
    completed = subprocess.run(
        command,
        cwd=cwd,
        check=False,
        env=os.environ.copy(),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()
print("PROJECT_DIR:", PROJECT_DIR)
print("kernel python:", sys.executable)
print("subprocess python:", PYTHON_EXE)
print("BUNDLE_DIR:", BUNDLE_DIR)


PROJECT_DIR: C:\Projects\mpb-latex-ocr
kernel python: c:\Projects\mpb-latex-ocr\.venv\Scripts\python.exe
subprocess python: C:\Projects\mpb-latex-ocr\.venv\Scripts\python.exe
BUNDLE_DIR: C:\Projects\mpb-latex-ocr\data\im2latex_sample_bundle


## Optional: Build The Bundle From A Local Kaggle Dataset Download

Use this when you already have `outputs/test_predictions.jsonl` locally but do not want to rerun the Kaggle training notebook. Set `DOWNLOAD_KAGGLE_DATASET = True` only if the dataset is not downloaded yet and your Kaggle API credentials are configured. Set `BUILD_BUNDLE_FROM_LOCAL_DATASET = True` to copy the exact referenced images into `data/im2latex_sample_bundle`.


In [2]:
LOCAL_FULL_PREDICTIONS_PATH = RUN_DIR / "test_predictions.jsonl"

if DOWNLOAD_KAGGLE_DATASET:
    LOCAL_KAGGLE_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    run_command([PYTHON_EXE, "-m", "pip", "install", "kaggle"])
    # Requires ~/.kaggle/kaggle.json or KAGGLE_USERNAME/KAGGLE_KEY.
    run_command([
        PYTHON_EXE,
        "-m",
        "kaggle",
        "datasets",
        "download",
        "-d",
        KAGGLE_DATASET_SLUG,
        "-p",
        LOCAL_KAGGLE_DATASET_DIR,
        "--unzip",
    ])

if BUILD_BUNDLE_FROM_LOCAL_DATASET:
    if not LOCAL_FULL_PREDICTIONS_PATH.exists():
        raise FileNotFoundError(f"Missing local predictions file: {LOCAL_FULL_PREDICTIONS_PATH}")
    if not LOCAL_KAGGLE_DATASET_DIR.exists():
        raise FileNotFoundError(f"Missing local dataset dir: {LOCAL_KAGGLE_DATASET_DIR}")

    run_module(
        "mpb_latex_ocr.export_prediction_samples",
        [
            "--predictions", LOCAL_FULL_PREDICTIONS_PATH,
            "--output-dir", BUNDLE_DIR,
            "--num-samples", DIRECT_BUNDLE_SAMPLE_COUNT,
            "--mode", DIRECT_BUNDLE_SAMPLE_MODE,
            "--seed", DIRECT_BUNDLE_SAMPLE_SEED,
            "--path-map", f"{KAGGLE_IMAGE_PATH_PREFIX}={LOCAL_KAGGLE_DATASET_DIR}",
            "--zip-out", RUN_DIR / "im2latex_sample_bundle.zip",
        ],
    )
else:
    print("Direct dataset bundle build is disabled. Toggle BUILD_BUNDLE_FROM_LOCAL_DATASET when needed.")

print("local dataset dir:", LOCAL_KAGGLE_DATASET_DIR)
print("bundle dir:", BUNDLE_DIR)


$ C:\Projects\mpb-latex-ocr\.venv\Scripts\python.exe -m mpb_latex_ocr.export_prediction_samples --predictions C:\Projects\mpb-latex-ocr\outputs\test_predictions.jsonl --output-dir C:\Projects\mpb-latex-ocr\data\im2latex_sample_bundle --num-samples 12000 --mode random --seed 42 --path-map /kaggle/input/datasets/gregoryeritsyan/im2latex-230k=C:\Projects\mpb-latex-ocr\data\kaggle\im2latex-230k --zip-out C:\Projects\mpb-latex-ocr\outputs\im2latex_sample_bundle.zip
{
  "source_predictions": "C:\\Projects\\mpb-latex-ocr\\outputs\\test_predictions.jsonl",
  "output_dir": "C:\\Projects\\mpb-latex-ocr\\data\\im2latex_sample_bundle",
  "mode": "random",
  "seed": 42,
  "requested_samples": 12000,
  "exported_samples": 11916,
  "missing_images": [],
  "manifest": "C:\\Projects\\mpb-latex-ocr\\data\\im2latex_sample_bundle\\manifest.csv",
  "predictions": "C:\\Projects\\mpb-latex-ocr\\data\\im2latex_sample_bundle\\predictions.jsonl",
  "zip": "C:\\Projects\\mpb-latex-ocr\\outputs\\im2latex_sample_b

## Load The Portable Bundle

The folder should contain `manifest.csv`, `predictions.jsonl`, `metadata.json`, and `images/`. If you downloaded the Kaggle zip to `outputs/im2latex_sample_bundle.zip`, extract it with PowerShell:

```powershell
Expand-Archive outputs\im2latex_sample_bundle.zip data\im2latex_sample_bundle -Force
```

In [3]:
if not BUNDLE_MANIFEST.exists() or not EXPORTED_PREDICTIONS_PATH.exists():
    raise FileNotFoundError(
        "Missing local Im2LaTeX bundle. Export it from Kaggle and extract it to "
        f"{BUNDLE_DIR}"
    )


def load_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def summarize_prediction_rows(rows: list[dict]) -> dict[str, float]:
    keys = [
        "exact_match",
        "norm_edit_distance",
        "render_f1",
        "render_iou",
        "render_match",
        "prediction_rendered",
        "target_rendered",
    ]
    summary = {"num_samples": len(rows)}
    for key in keys:
        if rows and key in rows[0]:
            summary[key] = sum(float(row.get(key, 0.0)) for row in rows) / len(rows)
    return summary


exported_rows = load_jsonl(EXPORTED_PREDICTIONS_PATH)
metadata = json.loads(BUNDLE_METADATA_PATH.read_text(encoding="utf-8")) if BUNDLE_METADATA_PATH.exists() else {}
print("metadata:")
print(json.dumps(metadata, indent=2))
print("exported Kaggle/server prediction summary:")
print(json.dumps(summarize_prediction_rows(exported_rows), indent=2))

try:
    import pandas as pd

    display(pd.DataFrame(exported_rows)[["sample_id", "target", "prediction", "exact_match", "norm_edit_distance", "render_f1"]].head(12))
except Exception:
    for row in exported_rows[:5]:
        print(row)


metadata:
{
  "source_predictions": "C:\\Projects\\mpb-latex-ocr\\outputs\\test_predictions.jsonl",
  "output_dir": "C:\\Projects\\mpb-latex-ocr\\data\\im2latex_sample_bundle",
  "mode": "random",
  "seed": 42,
  "requested_samples": 12000,
  "exported_samples": 11916,
  "missing_images": [],
  "manifest": "C:\\Projects\\mpb-latex-ocr\\data\\im2latex_sample_bundle\\manifest.csv",
  "predictions": "C:\\Projects\\mpb-latex-ocr\\data\\im2latex_sample_bundle\\predictions.jsonl"
}
exported Kaggle/server prediction summary:
{
  "num_samples": 11916,
  "exact_match": 0.3945115810674723,
  "norm_edit_distance": 0.10075292168449941,
  "render_f1": 0.6315401162754198,
  "render_iou": 0.5968360730073918,
  "render_match": 0.5115810674723061,
  "prediction_rendered": 0.8847767707284323,
  "target_rendered": 0.935464921114468
}


,sample_id,target,prediction,exact_match,norm_edit_distance,render_f1
0,116763,\sim e^{--}{\cal E}_{--}^{\underline{a}}e^{++}...,\sim e^{--}{\cal E}_{--}^{\underline{a}}e^{++}...,1.0,0.000000,0.000000
1,220783,{\cal J}_{i}^{\mu}=\frac{-1}{1+4{\Phi}^{2}}\bi...,{\cal J}_{i}^{\mu}=\frac{-1}{1+4\Phi^{2}}((\de...,0.0,0.077778,0.000000
2,150116,\Omega_{0}=-{\frac{1}{2}}\omega_{ij}\theta^{i}...,\Omega_{0}=-\frac{1}{2}\omega_{ij}\theta^{i}\w...,0.0,0.031746,1.000000
3,27324,"G(r,\theta)=\int g^{00}dg_{00}+2\int g^{03}dg_...","G(r,\theta)=\int g^{00}dg_{00}+2\int g^{03}dg_...",1.0,0.000000,1.000000
4,38777,"S=\int d^{2}x\,(B_{0}d\omega+B_{a}De^{a}-\sum_...","S=\int d^{2}x\,(B_{0}d\omega+B_{a}De^{a}-\sum_...",1.0,0.000000,1.000000
5,12983,\frac{d\tilde{K}_{0}(s)}{ds}=\tilde{K}_{2}(s)+...,\frac{d\tilde{K}_{0}(s)}{ds}=\tilde{K}_{2}(s)+...,0.0,0.020202,0.413560
6,215443,"{\sum_{i,j,k,l=1}^{h}\omega_{l}^{2}\partial_{z...","\sum_{i,j,k,l=1}^{h}\omega_{l}^{2}\partial_{z}...",0.0,0.025424,1.000000
7,34457,A_{M}^{\dagger}A_{M}=\tilde{A}_{M}^{\dagger}\t...,A_{M}^{\dagger}A_{M}=\tilde{A}_{M}^{\dagger}\t...,1.0,0.000000,1.000000
8,54712,S=\int d^{2n}x\[\dot{A}_{a[p]}E_{a[p]}-\frac{1...,S=\int d^{2n}x~[\dot{A}_{a[p]}E_{a[p]}-B_{a[p]...,0.0,0.383459,0.000000
9,180091,"\delta^{\alpha}z^{k}=\sum_{j}\{z^{k},z^{j}\}_{...","\delta^{\alpha}z^{k}=\sum_{j}\{z^{k},z^{j}\}_{...",0.0,0.010753,0.836166


## Rerun The Local Checkpoint On Exact Im2LaTeX Images

This uses the downloaded checkpoint in `outputs/checkpoints`, the downloaded tokenizer, and the portable bundle images. Disable `RUN_LOCAL_EVAL` in the first cell if you only want to inspect exported Kaggle predictions.

In [5]:
import argparse
from tqdm import tqdm

from mpb_latex_ocr.evaluate import evaluate as evaluate_checkpoint


def checkpoint_val_ned(path: Path) -> float | None:
    match = re.search(r"val_ned=([0-9]+(?:\.[0-9]+)?)", path.name)
    return float(match.group(1)) if match else None


def pick_checkpoint(checkpoint_dir: Path) -> Path:
    checkpoints = sorted(checkpoint_dir.glob("*.ckpt"))
    if not checkpoints:
        raise FileNotFoundError(f"No .ckpt files found under {checkpoint_dir}")
    with_metric = [(checkpoint_val_ned(path), path) for path in checkpoints if checkpoint_val_ned(path) is not None]
    if with_metric:
        return min(with_metric, key=lambda item: (item[0], item[1].name))[1]
    return max(checkpoints, key=lambda path: path.stat().st_mtime)


if RESOLVED_CONFIG_PATH.exists():
    resolved_config = json.loads(RESOLVED_CONFIG_PATH.read_text(encoding="utf-8"))
else:
    resolved_config = {}

data_config = resolved_config.get("data", {})
generation_config = resolved_config.get("generation", {})
IMAGE_HEIGHT = int(data_config.get("image_height", 128))
IMAGE_WIDTH = int(data_config.get("image_width", 512))
MAX_LABEL_LENGTH = int(data_config.get("max_label_length", 256))
MAX_GENERATION_LENGTH = int(generation_config.get("max_length", MAX_LABEL_LENGTH))
NUM_WORKERS = 0 if os.name == "nt" else int(data_config.get("num_workers", 2))
CHECKPOINT_PATH = pick_checkpoint(CHECKPOINT_DIR)

print("checkpoint:", CHECKPOINT_PATH)
print("tokenizer:", TOKENIZER_PATH)
print("image size:", IMAGE_WIDTH, "x", IMAGE_HEIGHT)

if RUN_LOCAL_EVAL:
    LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    args = argparse.Namespace(
        checkpoint=str(CHECKPOINT_PATH),
        manifest=str(BUNDLE_MANIFEST),
        image_root=str(BUNDLE_DIR),
        tokenizer=str(TOKENIZER_PATH),
        split="test",
        batch_size=16,
        num_workers=NUM_WORKERS,
        image_height=IMAGE_HEIGHT,
        image_width=IMAGE_WIDTH,
        max_label_length=MAX_LABEL_LENGTH,
        max_generation_length=MAX_GENERATION_LENGTH,
        max_batches=None,
        device="cuda" if __import__("torch").cuda.is_available() else "cpu",
        predictions_out=str(LOCAL_PREDICTIONS_PATH),
        render_metric=True,
        render_font_size=32,
        render_dpi=200,
        render_match_threshold=0.98,
        cdm_json_out=str(LOCAL_CDM_JSON_PATH),
    )

    metrics = evaluate_checkpoint(args)

    local_rows = load_jsonl(LOCAL_PREDICTIONS_PATH)
    print("local prediction summary:")
    print(json.dumps(summarize_prediction_rows(local_rows), indent=2))
    print("raw evaluation metrics:")
    print(json.dumps(metrics, indent=2))
else:
    local_rows = load_jsonl(LOCAL_PREDICTIONS_PATH) if LOCAL_PREDICTIONS_PATH.exists() else []
    print("local eval disabled")

checkpoint: C:\Projects\mpb-latex-ocr\outputs\checkpoints\epoch=041-val_ned=0.0746.ckpt
tokenizer: C:\Projects\mpb-latex-ocr\outputs\tokenizer.json
image size: 512 x 128


Evaluating test:   0%|          | 0/745 [00:00<?, ?it/s]c:\Projects\mpb-latex-ocr\.venv\Lib\site-packages\torch\nn\modules\activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
Evaluating test:   0%|          | 0/745 [00:02<?, ?it/s]


KeyboardInterrupt: 

## Official CDM Metrics

This cell uses `outputs/im2latex_bundle_check/cdm_predictions.json` for the local bundle. The bundle is intentionally small by default: `DIRECT_BUNDLE_SAMPLE_COUNT = 96` in the setup cell and the earlier export command used `--num-samples 96`. Increase that count, or export the full prediction file, if you want a larger local/CDM check.


In [ ]:
def safe_cdm_img_id(row: dict, index: int) -> str:
    raw_id = row.get("sample_id", row.get("image_id", row.get("img_id", f"sample_{index:06d}")))
    return re.sub(r"[^A-Za-z0-9_.=-]+", "_", str(raw_id))


def write_cdm_json_from_rows(rows: list[dict], output_path: Path) -> None:
    records = []
    for index, row in enumerate(rows):
        records.append(
            {
                "img_id": safe_cdm_img_id(row, index),
                "gt": row.get("target", ""),
                "pred": row.get("prediction", ""),
            }
        )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")


def official_cdm_metrics_path(input_json: Path, output_dir: Path) -> Path:
    return output_dir / input_json.stem / "metrics_res.json"


cdm_rows = local_rows if local_rows else exported_rows
if not cdm_rows:
    raise ValueError("No prediction rows are loaded. Run the bundle and local-eval cells first.")

if not LOCAL_CDM_JSON_PATH.exists():
    write_cdm_json_from_rows(cdm_rows, LOCAL_CDM_JSON_PATH)
    print("wrote CDM JSON:", LOCAL_CDM_JSON_PATH)

cdm_records = json.loads(LOCAL_CDM_JSON_PATH.read_text(encoding="utf-8"))
metrics_path = official_cdm_metrics_path(LOCAL_CDM_JSON_PATH, OFFICIAL_CDM_OUTPUT_DIR)
print("bundle rows:", len(exported_rows))
print("local rows:", len(local_rows))
print("CDM input records:", len(cdm_records))
print("CDM input:", LOCAL_CDM_JSON_PATH)
print("CDM metrics:", metrics_path)

should_run_cdm = RUN_OFFICIAL_CDM and (FORCE_RECOMPUTE_OFFICIAL_CDM or not metrics_path.exists())
if should_run_cdm:
    if os.name != "nt":
        raise RuntimeError("This notebook cell currently runs the Windows CDM helper. Use the WSL/Docker flow from wiki.md on Linux/macOS.")
    run_command(
        [
            "powershell",
            "-ExecutionPolicy", "Bypass",
            "-File", PROJECT_DIR / "scripts" / "cdm" / "run_official_cdm_windows.ps1",
            "-InputJson", LOCAL_CDM_JSON_PATH,
            "-OutputDir", OFFICIAL_CDM_OUTPUT_DIR,
            "-Pools", OFFICIAL_CDM_POOLS,
        ]
    )
elif not metrics_path.exists():
    print("Official CDM metrics are missing. Set RUN_OFFICIAL_CDM=True to compute them.")

if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    details = metrics.get("details", {})
    input_ids = [str(record.get("img_id", f"sample_{index}")) for index, record in enumerate(cdm_records)]
    missing_ids = sorted(set(input_ids) - set(details))
    summary = {
        "input_records": len(cdm_records),
        "scored_records": len(details),
        "unscored_records": len(missing_ids),
        "mean_score": metrics.get("mean_score"),
        "exp_rate": metrics.get("exp_rate"),
        "metrics_path": str(metrics_path),
    }
    print("official CDM summary:")
    print(json.dumps(summary, indent=2))

    detail_rows = [
        {"img_id": img_id, **score}
        for img_id, score in details.items()
    ]
    try:
        import pandas as pd

        display(pd.DataFrame([summary]))
        if detail_rows:
            detail_df = pd.DataFrame(detail_rows).sort_values(["F1_score", "recall", "precision"])
            display(detail_df.head(20))
        if missing_ids:
            display(pd.DataFrame({"unscored_img_id": missing_ids}))
    except Exception:
        print("worst CDM rows:")
        for item in sorted(detail_rows, key=lambda row: row.get("F1_score", 0.0))[:20]:
            print(item)
        if missing_ids:
            print("unscored ids:", missing_ids)
else:
    print("No official CDM metrics available yet.")


## Visual Spot Check

Set `VISUAL_SOURCE` to `exported` to inspect the predictions produced on Kaggle, or `local` to inspect the predictions rerun on this machine.

In [ ]:
from mpb_latex_ocr.metrics.render import render_formula_image

import re

VISUAL_SOURCE = "local" if RUN_LOCAL_EVAL and local_rows else "exported"
visual_rows = local_rows if VISUAL_SOURCE == "local" else exported_rows
VISUAL_OUTPUT_DIR = LOCAL_OUTPUT_DIR / "visual_examples"
VISUAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def safe_formula_image(formula: str) -> tuple[Image.Image, str | None]:
    try:
        return render_formula_image(formula), None
    except Exception as exc:
        image = Image.new("L", (900, 180), color=255)
        draw = ImageDraw.Draw(image)
        draw.text(
            (12, 12),
            f"render failed: {type(exc).__name__}\n{str(formula)[:180]}",
            fill=0,
        )
        return image, f"{type(exc).__name__}: {exc}"


def clipped_text(value: str, max_chars: int = 1600) -> str:
    value = str(value)
    if len(value) > max_chars:
        value = value[:max_chars] + " ..."
    return "\n".join(textwrap.wrap(value, width=128, break_long_words=True))


def resolve_bundle_image_path(row: dict) -> Path:
    image_path = Path(str(row["image_path"]))
    if image_path.is_absolute():
        return image_path
    return BUNDLE_DIR / image_path


def draw_label(axis, label: str, detail: str = "", y: float = 0.5, va: str = "center") -> None:
    axis.axis("off")
    axis.text(
        0.98,
        y,
        label,
        va=va,
        ha="right",
        fontsize=12,
        fontweight="bold",
        color="#111827",
        transform=axis.transAxes,
    )
    if detail:
        axis.text(
            0.98,
            y - 0.12,
            detail,
            va="top",
            ha="right",
            fontsize=8.5,
            color="#4b5563",
            transform=axis.transAxes,
        )


def display_visual_example(row: dict, index: int) -> Path:
    input_image = Image.open(resolve_bundle_image_path(row)).convert("L")
    target_image, target_error = safe_formula_image(row.get("target", ""))
    prediction_image, prediction_error = safe_formula_image(row.get("prediction", ""))

    render_f1 = float(row.get("render_f1", 0.0))
    render_iou = float(row.get("render_iou", 0.0))
    exact_match = row.get("exact_match", "")
    normalized_edit_distance = row.get("norm_edit_distance", row.get("normalized_edit_distance", ""))
    image_id = row.get("sample_id", row.get("image_id", row.get("img_id", "unknown")))
    safe_image_id = re.sub(r"[^A-Za-z0-9_.=-]+", "_", str(image_id))

    fig = plt.figure(figsize=(15, 14.5), constrained_layout=True)
    grid = fig.add_gridspec(
        4,
        2,
        width_ratios=[0.16, 1.0],
        height_ratios=[1.0, 1.2, 1.2, 3.4],
        hspace=0.06,
        wspace=0.03,
    )
    label_axes = [fig.add_subplot(grid[row_index, 0]) for row_index in range(4)]
    content_axes = [fig.add_subplot(grid[row_index, 1]) for row_index in range(4)]

    draw_label(label_axes[0], "input", f"sample {index:02d}\nimage_id={image_id}")
    draw_label(label_axes[1], "target", "render failed" if target_error else "")
    draw_label(label_axes[2], "prediction", f"f1={render_f1:.3f}\niou={render_iou:.3f}" + ("\nrender failed" if prediction_error else ""))
    draw_label(label_axes[3], "raw", f"exact={exact_match}\nned={normalized_edit_distance}", y=0.98, va="top")

    for axis, image in [
        (content_axes[0], input_image),
        (content_axes[1], target_image),
        (content_axes[2], prediction_image),
    ]:
        axis.imshow(image, cmap="gray", vmin=0, vmax=255)
        axis.axis("off")

    render_error_text = row.get("render_error") or ""
    raw_text = (
        f"target:\n{clipped_text(row.get('target', ''))}\n\n"
        f"prediction:\n{clipped_text(row.get('prediction', ''))}"
        + (f"\n\nrender_error:\n{clipped_text(render_error_text, 900)}" if render_error_text else "")
    )
    content_axes[3].axis("off")
    content_axes[3].set_facecolor("#f8fafc")
    content_axes[3].patch.set_visible(True)
    content_axes[3].text(
        0.0,
        1.0,
        raw_text,
        va="top",
        ha="left",
        family="monospace",
        fontsize=9.5,
        linespacing=1.28,
        transform=content_axes[3].transAxes,
    )

    output_path = VISUAL_OUTPUT_DIR / f"sample_{index:02d}_{safe_image_id}.png"
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    return output_path


rows = list(visual_rows)
if VISUAL_SAMPLE_MODE == "random":
    import random

    random.Random(VISUAL_SEED).shuffle(rows)
elif VISUAL_SAMPLE_MODE == "worst_render":
    rows.sort(key=lambda row: float(row.get("render_f1", 0.0)))
elif VISUAL_SAMPLE_MODE == "best_render":
    rows.sort(key=lambda row: float(row.get("render_f1", 0.0)), reverse=True)
elif VISUAL_SAMPLE_MODE != "first":
    raise ValueError(f"Unknown VISUAL_SAMPLE_MODE: {VISUAL_SAMPLE_MODE}")

sample_rows = rows[: min(VISUAL_SAMPLE_COUNT, len(rows))]
print("visual source:", VISUAL_SOURCE)
print("visual examples:", len(sample_rows))
print("saving per-example figures to:", VISUAL_OUTPUT_DIR)

saved_paths = [display_visual_example(row, index) for index, row in enumerate(sample_rows, start=1)]
print("saved files:")
for saved_path in saved_paths:
    print("-", saved_path)
